# Nanochat PPO and PPO-Standard Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and runs two Kaggle-friendly PPO variants:

- `scripts.chat_ppo`
- `scripts.chat_ppo_standard`

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` expected)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes PPO checkpoints under `chatppo_checkpoints/d8_kaggle` and `chatppo_standard_checkpoints/d8_kaggle`.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


Code input: /kaggle/input/datasets/jennyqqjiang/nanochat-codes
SFT cache input: /kaggle/input/datasets/yshuaiqin/nanochat-d8-sft-cache
Working repo: /kaggle/working/nanochat
Nanochat cache: /kaggle/working/nanochat_cache
HF cache: /kaggle/working/huggingface_cache
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
2.1G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Install Dependencies


In [2]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'requests>=2.32.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
    'rustbpe>=0.1.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.2 MB/s eta 0:00:00
Dependencies installed


## Configure Runtime


In [3]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
env['PYTHONUNBUFFERED'] = '1'

# PPO scripts do not use GradScaler, so float32 is safer on T4.
env['NANOCHAT_DTYPE'] = 'float32'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
env['NCCL_DEBUG'] = 'WARN'

os.environ.update(env)

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Validate Repo And SFT Cache


In [4]:
assert (WORK_REPO / 'scripts' / 'chat_ppo.py').exists(), 'Missing scripts/chat_ppo.py'
assert (WORK_REPO / 'scripts' / 'chat_ppo_standard.py').exists(), 'Missing scripts/chat_ppo_standard.py'
assert (WORK_REPO / 'scripts' / 'chat_post_eval.py').exists(), 'Missing scripts/chat_post_eval.py'
assert (WORK_REPO / 'scripts' / 'chat_web.py').exists(), 'Missing scripts/chat_web.py'
assert (WORK_REPO / 'nanochat' / 'checkpoint_manager.py').exists(), 'Missing nanochat/checkpoint_manager.py'

MODEL_TAG = 'd8_kaggle'
sft_dir = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
tokenizer_dir = WORK_CACHE / 'tokenizer'
assert sft_dir.exists(), f'Missing SFT checkpoint dir: {sft_dir}'
assert tokenizer_dir.exists(), f'Missing tokenizer dir: {tokenizer_dir}'
assert sorted(sft_dir.glob('model_*.pt')), f'No model_*.pt files found in {sft_dir}'
assert sorted(sft_dir.glob('meta_*.json')), f'No meta_*.json files found in {sft_dir}'

subprocess.check_call(
    [
        sys.executable, '-m', 'py_compile',
        'scripts/chat_ppo.py',
        'scripts/chat_ppo_standard.py',
        'scripts/chat_post_eval.py',
        'scripts/chat_web.py',
        'nanochat/checkpoint_manager.py',
    ],
    cwd=WORK_REPO,
    env=env,
)

print('Setup complete')
print('SFT checkpoints:', [p.name for p in sorted(sft_dir.glob('model_*.pt'))[-5:]])
print('Tokenizer files:', sorted(p.name for p in tokenizer_dir.iterdir()))


Setup complete
SFT checkpoints: ['model_004999.pt']
Tokenizer files: ['token_bytes.pt', 'tokenizer.pkl']


## PPO Run


In [5]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
MODEL_TAG = 'd8_kaggle'
OVERWRITE_PPO_CHECKPOINTS = True

if OVERWRITE_PPO_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatppo_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatppo_reward_checkpoints' / MODEL_TAG, ignore_errors=True)

ppo_args = [
    '-m', 'scripts.chat_ppo',

    '--run=dummy',

    '--policy-source=sft',
    f'--policy-tag={MODEL_TAG}',
    '--reference-source=sft',
    f'--reference-tag={MODEL_TAG}',

    '--preference-source=gsm8k',
    '--max-train-examples=4096',
    '--max-val-examples=512',

    '--rm-epochs=2',
    '--rm-batch-size=4',
    '--rm-lr=1e-4',
    '--rm-train-backbone=0',

    '--ppo-steps=300',
    '--ppo-epochs=1',
    '--prompt-batch-size=4',
    '--device-batch-size=2',
    '--max-new-tokens=256',

    '--temperature=1.0',
    '--top-k=0',

    '--clip-epsilon=0.2',
    '--kl-beta=0.02',
    '--kl-reduction=sum',
    '--advantage-whiten=1',

    '--embedding-lr=0.03',
    '--unembedding-lr=0.0008',
    '--matrix-lr=0.002',
    '--init-lr-frac=0.05',
    '--weight-decay=0.0',

    '--save-every=100',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + ppo_args
else:
    cmd = [sys.executable] + ppo_args

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

policy_dir = WORK_CACHE / 'chatppo_checkpoints' / MODEL_TAG
reward_dir = WORK_CACHE / 'chatppo_reward_checkpoints' / MODEL_TAG

print('PPO policy checkpoints:', [p.name for p in sorted(policy_dir.glob('model_*.pt'))])
print('PPO reward checkpoints:', [p.name for p in sorted(reward_dir.glob('reward_*.pt'))])


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_ppo --run=dummy --policy-source=sft --policy-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --preference-source=gsm8k --max-train-examples=4096 --max-val-examples=512 --rm-epochs=2 --rm-batch-size=4 --rm-lr=1e-4 --rm-train-backbone=0 --ppo-steps=300 --ppo-epochs=1 --prompt-batch-size=4 --device-batch-size=2 --max-new-tokens=256 --temperature=1.0 --top-k=0 --clip-epsilon=0.2 --kl-beta=0.02 --kl-reduction=sum --advantage-whiten=1 --embedding-lr=0.03 --unembedding-lr=0.0008 --matrix-lr=0.002 --init-lr-frac=0.05 --weight-decay=0.0 --save-every=100


[W614 04:31:44.607431281 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 04:31:57,263 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:31:57,263 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:31:57,266 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 04:31:57,266 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 04:31:58.545376955 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 04:31:58.547521464 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Saved PPO checkpoints to /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle and /kaggle/working/nanochat_cache/chatppo_reward_checkpoints/d8_kaggle
Step 101 | ppo_loss: 0.514517 | reward: -0.0057 | return: -0.8975 | sampled_kl: 44.5893 | ratio: 1.0000 | clipfrac: 0.0000
Step 102 | ppo_loss: -0.036004 | reward: 0.3283 | return: -1.2299 | sampled_kl: 77.9114 | ratio: 1.0000 | clipfrac: 0.0000
Step 103 | ppo_loss: -0.010122 | reward: 0.0541 | return: -0.9159 | sampled_kl: 48.5009 | ratio: 1.0000 | clipfrac: 0.0000
Step 104 | ppo_loss: 0.181948 | reward: 0.1426 | return: -0.8132 | sampled_kl: 47.7923 | ratio: 1.0000 | clipfrac: 0.0000
Step 105 | ppo_loss: 0.147409 | reward: 0.0375 | return: -1.0000 | sampled_kl: 51.8717 | ratio: 1.0000 | clipfrac: 0.0000
Step 106 | ppo_loss: 0.201005 | reward: 0.4227 | return: -0.0971 | sampled_kl: 25.9863 | ratio: 1.0000 | clipfrac: 0.0000
Step 107 | ppo_loss: -0.125893 | reward: 0.3527 | return: -0.6948 | sampled_kl: 52.3743 | ratio: 1.0000 | c

2026-06-14 04:57:18,414 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle/model_000200.pt
2026-06-14 04:57:18,414 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle/meta_000200.json


Saved PPO checkpoints to /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle and /kaggle/working/nanochat_cache/chatppo_reward_checkpoints/d8_kaggle
Step 201 | ppo_loss: -0.434788 | reward: 0.9267 | return: 0.2985 | sampled_kl: 31.4117 | ratio: 1.0000 | clipfrac: 0.0000
Step 202 | ppo_loss: -0.222774 | reward: 0.2349 | return: -0.2414 | sampled_kl: 23.8165 | ratio: 1.0000 | clipfrac: 0.0000
Step 203 | ppo_loss: -0.341275 | reward: 0.7648 | return: 0.1795 | sampled_kl: 29.2652 | ratio: 1.0000 | clipfrac: 0.0000
Step 204 | ppo_loss: -0.319393 | reward: 0.7126 | return: 0.1654 | sampled_kl: 27.3601 | ratio: 1.0000 | clipfrac: 0.0000
Step 205 | ppo_loss: -0.155418 | reward: 1.8570 | return: 1.4153 | sampled_kl: 22.0855 | ratio: 1.0000 | clipfrac: 0.0000
Step 206 | ppo_loss: -0.211340 | reward: 1.0264 | return: 0.4177 | sampled_kl: 30.4351 | ratio: 1.0000 | clipfrac: 0.0000
Step 207 | ppo_loss: -0.271434 | reward: 2.8792 | return: 2.2400 | sampled_kl: 31.9587 | ratio: 1.0000 | clip

2026-06-14 05:12:07,578 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle/model_000300.pt
2026-06-14 05:12:07,578 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle/meta_000300.json


Saved final PPO checkpoints to /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle and /kaggle/working/nanochat_cache/chatppo_reward_checkpoints/d8_kaggle
PPO policy checkpoints: ['model_000100.pt', 'model_000200.pt', 'model_000300.pt']
PPO reward checkpoints: ['reward_000100.pt', 'reward_000200.pt', 'reward_000300.pt']


## PPO-Standard Run


In [6]:
MODEL_TAG = 'd8_kaggle'
OVERWRITE_PPO_STANDARD_CHECKPOINTS = True

if OVERWRITE_PPO_STANDARD_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatppo_standard_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatppo_standard_reward_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatppo_standard_value_checkpoints' / MODEL_TAG, ignore_errors=True)

ppo_standard_env = env.copy()
ppo_standard_env['NANOCHAT_DTYPE'] = 'float32'

ppo_standard_args = [
    '-m', 'scripts.chat_ppo_standard',

    '--run=dummy',

    '--policy-source=sft',
    f'--policy-tag={MODEL_TAG}',
    '--reference-source=sft',
    f'--reference-tag={MODEL_TAG}',

    '--preference-source=gsm8k',
    '--max-train-examples=4096',
    '--max-val-examples=512',

    '--rm-epochs=2',
    '--rm-batch-size=4',
    '--rm-lr=1e-4',
    '--rm-train-backbone=0',

    '--ppo-steps=300',
    '--ppo-epochs=2',
    '--prompt-batch-size=2',
    '--device-batch-size=2',
    '--ppo-minibatch-size=2',
    '--max-new-tokens=256',

    '--temperature=1.0',
    '--top-k=0',

    '--clip-epsilon=0.2',
    '--value-clip-epsilon=0.2',
    '--kl-beta=0.02',
    '--gamma=1.0',
    '--gae-lambda=0.95',
    '--advantage-whiten=1',

    '--value-loss-coef=0.25',
    '--entropy-coef=0.0',
    '--value-lr=1e-4',
    '--value-train-backbone=1',

    '--embedding-lr=0.02',
    '--unembedding-lr=0.0005',
    '--matrix-lr=0.0015',
    '--init-lr-frac=0.05',
    '--weight-decay=0.0',

    '--save-every=100',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + ppo_standard_args
else:
    cmd = [sys.executable] + ppo_standard_args

print('Running command:')
print(' '.join(map(str, cmd)))
subprocess.run(list(map(str, cmd)), cwd=WORK_REPO, env=ppo_standard_env, check=True)

policy_dir = WORK_CACHE / 'chatppo_standard_checkpoints' / MODEL_TAG
reward_dir = WORK_CACHE / 'chatppo_standard_reward_checkpoints' / MODEL_TAG
value_dir = WORK_CACHE / 'chatppo_standard_value_checkpoints' / MODEL_TAG

print('PPO-standard policy checkpoints:', [p.name for p in sorted(policy_dir.glob('model_*.pt'))])
print('PPO-standard reward checkpoints:', [p.name for p in sorted(reward_dir.glob('reward_*.pt'))])
print('PPO-standard value checkpoints:', [p.name for p in sorted(value_dir.glob('value_*.pt'))])


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_ppo_standard --run=dummy --policy-source=sft --policy-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --preference-source=gsm8k --max-train-examples=4096 --max-val-examples=512 --rm-epochs=2 --rm-batch-size=4 --rm-lr=1e-4 --rm-train-backbone=0 --ppo-steps=300 --ppo-epochs=2 --prompt-batch-size=2 --device-batch-size=2 --ppo-minibatch-size=2 --max-new-tokens=256 --temperature=1.0 --top-k=0 --clip-epsilon=0.2 --value-clip-epsilon=0.2 --kl-beta=0.02 --gamma=1.0 --gae-lambda=0.95 --advantage-whiten=1 --value-loss-coef=0.25 --entropy-coef=0.0 --value-lr=1e-4 --value-train-backbone=1 --embedding-lr=0.02 --unembedding-lr=0.0005 --matrix-lr=0.0015 --init-lr-frac=0.05 --weight-decay=0.0 --save-every=100


[W614 05:12:14.693098769 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 05:12:20,038 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:12:20,039 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 05:12:20,051 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:12:20,052 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 05:12:20.343408335 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 05:12:20.351183010 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Saved standard PPO checkpoints to /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle, /kaggle/working/nanochat_cache/chatppo_standard_reward_checkpoints/d8_kaggle, and /kaggle/working/nanochat_cache/chatppo_standard_value_checkpoints/d8_kaggle
Step 101 | pg_loss: -0.002681 | value_loss: 0.044487 | reward: -0.4859 | return: -0.8140 | ref_kl: 16.4084 | ratio: 0.9999 | clipfrac: 0.0000
Step 102 | pg_loss: -0.003738 | value_loss: 0.040553 | reward: -0.7884 | return: -1.2054 | ref_kl: 20.8526 | ratio: 1.0001 | clipfrac: 0.0000
Step 103 | pg_loss: -0.004546 | value_loss: 0.049493 | reward: -0.7476 | return: -1.0186 | ref_kl: 13.5482 | ratio: 0.9995 | clipfrac: 0.0000
Step 104 | pg_loss: -0.003076 | value_loss: 0.061977 | reward: -0.4485 | return: -0.7094 | ref_kl: 13.0423 | ratio: 1.0000 | clipfrac: 0.0000
Step 105 | pg_loss: -0.003631 | value_loss: 0.049230 | reward: 0.0478 | return: -0.2384 | ref_kl: 14.3075 | ratio: 0.9999 | clipfrac: 0.0000
Step 106 | pg_loss: -0.00305

2026-06-14 05:31:02,113 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle/model_000200.pt
2026-06-14 05:31:02,114 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle/meta_000200.json


Saved standard PPO checkpoints to /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle, /kaggle/working/nanochat_cache/chatppo_standard_reward_checkpoints/d8_kaggle, and /kaggle/working/nanochat_cache/chatppo_standard_value_checkpoints/d8_kaggle
Step 201 | pg_loss: -0.001845 | value_loss: 0.054687 | reward: -1.1244 | return: -1.5685 | ref_kl: 22.2069 | ratio: 0.9999 | clipfrac: 0.0000
Step 202 | pg_loss: -0.002008 | value_loss: 0.052640 | reward: -0.4238 | return: -0.8409 | ref_kl: 20.8567 | ratio: 0.9999 | clipfrac: 0.0000
Step 203 | pg_loss: -0.001965 | value_loss: 0.050946 | reward: -0.6445 | return: -1.0284 | ref_kl: 19.1959 | ratio: 1.0000 | clipfrac: 0.0000
Step 204 | pg_loss: -0.002156 | value_loss: 0.043785 | reward: -0.5106 | return: -0.8814 | ref_kl: 18.5398 | ratio: 0.9999 | clipfrac: 0.0000
Step 205 | pg_loss: -0.002055 | value_loss: 0.049035 | reward: 0.0028 | return: -0.4132 | ref_kl: 20.7973 | ratio: 0.9999 | clipfrac: 0.0000
Step 206 | pg_loss: -0.00218

2026-06-14 05:39:44,781 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle/model_000300.pt
2026-06-14 05:39:44,786 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle/meta_000300.json


Saved standard PPO checkpoints to /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle, /kaggle/working/nanochat_cache/chatppo_standard_reward_checkpoints/d8_kaggle, and /kaggle/working/nanochat_cache/chatppo_standard_value_checkpoints/d8_kaggle
PPO-standard policy checkpoints: ['model_000100.pt', 'model_000200.pt', 'model_000300.pt']
PPO-standard reward checkpoints: ['reward_000100.pt', 'reward_000200.pt', 'reward_000300.pt']
PPO-standard value checkpoints: ['value_000100.pt', 'value_000200.pt', 'value_000300.pt']


## Inspect PPO Checkpoints


In [7]:
families = [
    'chatppo_checkpoints',
    'chatppo_reward_checkpoints',
    'chatppo_standard_checkpoints',
    'chatppo_standard_reward_checkpoints',
    'chatppo_standard_value_checkpoints',
]
for family in families:
    checkpoint_dir = WORK_CACHE / family / MODEL_TAG
    print()
    print(family, checkpoint_dir)
    print('Exists:', checkpoint_dir.exists())
    if checkpoint_dir.exists():
        print('Files:', [p.name for p in sorted(checkpoint_dir.glob('*'))])

subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)



chatppo_checkpoints /kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle
Exists: True
Files: ['meta_000100.json', 'meta_000200.json', 'meta_000300.json', 'model_000100.pt', 'model_000200.pt', 'model_000300.pt']

chatppo_reward_checkpoints /kaggle/working/nanochat_cache/chatppo_reward_checkpoints/d8_kaggle
Exists: True
Files: ['reward_000100.pt', 'reward_000200.pt', 'reward_000300.pt']

chatppo_standard_checkpoints /kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle
Exists: True
Files: ['meta_000100.json', 'meta_000200.json', 'meta_000300.json', 'model_000100.pt', 'model_000200.pt', 'model_000300.pt']

chatppo_standard_reward_checkpoints /kaggle/working/nanochat_cache/chatppo_standard_reward_checkpoints/d8_kaggle
Exists: True
Files: ['reward_000100.pt', 'reward_000200.pt', 'reward_000300.pt']

chatppo_standard_value_checkpoints /kaggle/working/nanochat_cache/chatppo_standard_value_checkpoints/d8_kaggle
Exists: True
Files: ['value_000100.pt', 'value_000200.pt',

CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Comparison Eval


In [8]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 100

if RUN_COMPARISON_EVAL:
    models = ['sft=sft:d8_kaggle']

    ppo_dir = WORK_CACHE / 'chatppo_checkpoints'
    if (ppo_dir / MODEL_TAG).exists():
        models.append(f'ppo=@{ppo_dir}:{MODEL_TAG}')

    ppo_standard_dir = WORK_CACHE / 'chatppo_standard_checkpoints'
    if (ppo_standard_dir / MODEL_TAG).exists():
        models.append(f'ppo_standard=@{ppo_standard_dir}:{MODEL_TAG}')

    post_eval_args = [
        '-m', 'scripts.chat_post_eval',
    ]
    for model in models:
        post_eval_args.extend(['--models', model])
    post_eval_args.extend([
        '--max-problems', str(EVAL_MAX_PROBLEMS),
        '--batch-size', '2',
    ])

    if NPROC > 1:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args

    print('Running command:')
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print('Skipping comparison eval')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_post_eval --models sft=sft:d8_kaggle --models ppo=@/kaggle/working/nanochat_cache/chatppo_checkpoints:d8_kaggle --models ppo_standard=@/kaggle/working/nanochat_cache/chatppo_standard_checkpoints:d8_kaggle --max-problems 100 --batch-size 2


[W614 05:39:50.701262419 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 05:39:54,882 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:39:54,884 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 05:39:54,888 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:39:54,890 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 05:39:55.523496363 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 05:39:55.533718752 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Final: 20/100 (20.00%)
ppo_standard | MMLU: 20.00%


2026-06-14 05:57:28,794 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:57:28,797 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 05:57:28,809 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:57:28,812 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 05:57:28,849 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 05:57:28,856 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k

Rank 0 | 0/50 (0.00%)
Rank 1 | 0/50 (0.00%)
Final: 0/100 (0.00%)
ppo_standard | GSM8K: 0.00%


2026-06-14 06:00:49,521 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 06:00:49,537 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 06:00:49,578 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/openai_humaneval.py "HTTP/1.1 404 Not Found"
2026-06-14 06:00:49,580 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 06:00:49,596 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 06:00:49,638 - httpx - INFO

Rank 0 | 0/50 (0.00%)
Rank 1 | 0/50 (0.00%)
Final: 0/100 (0.00%)
ppo_standard | HumanEval: 0.00%
Rank 1 | 47/50 (94.00%)
Rank 0 | 43/50 (86.00%)
Final: 90/100 (90.00%)
ppo_standard | SpellingBee: 90.00%

Model        | ARC-Easy | ARC-Challenge | MMLU   | GSM8K | HumanEval | SpellingBee | Mean  
-------------+----------+---------------+--------+-------+-----------+-------------+-------
sft          | 26.00%   | 29.00%        | 24.00% | 0.00% | 0.00%     | 95.00%      | 29.00%
ppo          | 24.00%   | 31.00%        | 18.00% | 0.00% | 0.00%     | 11.00%      | 14.00%
ppo_standard | 20.00%   | 29.00%        | 20.00% | 0.00% | 0.00%     | 90.00%      | 26.50%

Ranking by mean score:
1. sft: 29.00%
2. ppo_standard: 26.50%
3. ppo: 14.00%


## Inspect Saved Comparison Report


In [9]:
report_path = WORK_CACHE / 'report' / 'chat-post-eval.md'
print(report_path)
print('Exists:', report_path.exists())
if report_path.exists():
    print(report_path.read_text())


/kaggle/working/nanochat_cache/report/chat-post-eval.md
Exists: True
## Chat Post Eval
timestamp: 2026-06-14 06:05:32

- tasks: ['ARC-Easy', 'ARC-Challenge', 'MMLU', 'GSM8K', 'HumanEval', 'SpellingBee']
- num_samples: 1
- temperature: 0.0000
- max_new_tokens: 512
- batch_size: 2
- max_problems: 100
- models: [{'label': 'sft', 'origin': 'sft', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 4999, 'resolved_meta_keys': ['model_config', 'step', 'user_config', 'val_bpb']}, {'label': 'ppo', 'origin': '/kaggle/working/nanochat_cache/chatppo_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 300, 'resolved_meta_keys': ['model_config']}, {'label': 'ppo_standard', 'origin': '/kaggle/working/nanochat_cache/chatppo_standard_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle',

## Output Manifest


In [10]:
manifest = {
    'model_tag': MODEL_TAG,
    'sft_checkpoint_dir': str(WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG),
    'ppo_checkpoint_dir': str(WORK_CACHE / 'chatppo_checkpoints' / MODEL_TAG),
    'ppo_reward_checkpoint_dir': str(WORK_CACHE / 'chatppo_reward_checkpoints' / MODEL_TAG),
    'ppo_standard_checkpoint_dir': str(WORK_CACHE / 'chatppo_standard_checkpoints' / MODEL_TAG),
    'ppo_standard_reward_checkpoint_dir': str(WORK_CACHE / 'chatppo_standard_reward_checkpoints' / MODEL_TAG),
    'ppo_standard_value_checkpoint_dir': str(WORK_CACHE / 'chatppo_standard_value_checkpoints' / MODEL_TAG),
    'report': str(WORK_CACHE / 'report' / 'chat-post-eval.md'),
}
manifest_path = Path('/kaggle/working/nanochat_ppo_ppostandard_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())


/kaggle/working/nanochat_ppo_ppostandard_manifest.json
{
  "model_tag": "d8_kaggle",
  "sft_checkpoint_dir": "/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle",
  "ppo_checkpoint_dir": "/kaggle/working/nanochat_cache/chatppo_checkpoints/d8_kaggle",
  "ppo_reward_checkpoint_dir": "/kaggle/working/nanochat_cache/chatppo_reward_checkpoints/d8_kaggle",
  "ppo_standard_checkpoint_dir": "/kaggle/working/nanochat_cache/chatppo_standard_checkpoints/d8_kaggle",
  "ppo_standard_reward_checkpoint_dir": "/kaggle/working/nanochat_cache/chatppo_standard_reward_checkpoints/d8_kaggle",
  "ppo_standard_value_checkpoint_dir": "/kaggle/working/nanochat_cache/chatppo_standard_value_checkpoints/d8_kaggle",
  "report": "/kaggle/working/nanochat_cache/report/chat-post-eval.md"
}


In [11]:
# Optional: save the PPO/PPO-standard checkpoint cache as a Kaggle Dataset.
import json

PPO_MODEL_TAG = MODEL_TAG
PPO_CACHE_DIR = Path('/kaggle/working/nanochat_d8_ppo_ppostandard_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-ppo-ppostandard-cache'
TITLE = 'nanochat d8 ppo ppostandard cache'
VERSION_MESSAGE = f'Save {PPO_MODEL_TAG} PPO and PPO-standard checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

required_dirs = [
    WORK_CACHE / 'chatppo_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_reward_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_standard_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_standard_reward_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_standard_value_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'tokenizer',
]
for required_dir in required_dirs:
    assert required_dir.exists(), f'Missing required directory: {required_dir}'

assert sorted((WORK_CACHE / 'chatppo_checkpoints' / PPO_MODEL_TAG).glob('model_*.pt')), 'No PPO policy checkpoints found'
assert sorted((WORK_CACHE / 'chatppo_standard_checkpoints' / PPO_MODEL_TAG).glob('model_*.pt')), 'No PPO-standard policy checkpoints found'

if PPO_CACHE_DIR.exists():
    shutil.rmtree(PPO_CACHE_DIR)
PPO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for family in [
    'chatppo_checkpoints',
    'chatppo_reward_checkpoints',
    'chatppo_standard_checkpoints',
    'chatppo_standard_reward_checkpoints',
    'chatppo_standard_value_checkpoints',
]:
    shutil.copytree(WORK_CACHE / family, PPO_CACHE_DIR / family)
shutil.copytree(WORK_CACHE / 'tokenizer', PPO_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = PPO_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(PPO_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(PPO_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(PPO_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Folder size:
5.7G	/kaggle/working/nanochat_d8_ppo_ppostandard_cache
Running: kaggle datasets create -p /kaggle/working/nanochat_d8_ppo_ppostandard_cache --dir-mode tar
Starting upload for file chatppo_reward_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:13<00:00, 111MB/s]


Upload successful: chatppo_reward_checkpoints.tar (1GB)
Starting upload for file chatppo_standard_value_checkpoints.tar


100%|██████████| 30.0k/30.0k [00:00<00:00, 163kB/s]


Upload successful: chatppo_standard_value_checkpoints.tar (30KB)
Starting upload for file chatppo_checkpoints.tar


...

Upload successful: tokenizer.tar (540KB)
Starting upload for file chatppo_standard_reward_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:18<00:00, 81.5MB/s]


Upload successful: chatppo_standard_reward_checkpoints.tar (1GB)
Starting upload for file chatppo_standard_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:17<00:00, 88.2MB/s]


Upload successful: chatppo_standard_checkpoints.tar (1GB)
Dataset creation error: Dataset url's dataset slugs and hashlink are all null
Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-ppo-ppostandard-cache


## Serve the PPO/PPO-standard Chat Model

Run this after `chatppo_checkpoints/d8_kaggle` or `chatppo_standard_checkpoints/d8_kaggle` exists. Set `SERVE_SOURCE` to `ppo` or `ppo_standard` before running the first cell.


In [ ]:
import time
import requests

SERVE_SOURCE = 'ppo_standard'  # choose 'ppo' or 'ppo_standard'
SERVE_MODEL_TAG = MODEL_TAG
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'

checkpoint_family = {
    'ppo': 'chatppo_checkpoints',
    'ppo_standard': 'chatppo_standard_checkpoints',
}[SERVE_SOURCE]
CHECKPOINT_DIR = WORK_CACHE / checkpoint_family

model_dir = CHECKPOINT_DIR / SERVE_MODEL_TAG
assert model_dir.exists(), f'Missing {SERVE_SOURCE} checkpoint directory: {model_dir}'
assert sorted(model_dir.glob('model_*.pt')), f'No model_*.pt files found in {model_dir}'
assert sorted(model_dir.glob('meta_*.json')), f'No meta_*.json files found in {model_dir}'

try:
    if server.poll() is None:
        print('Stopping existing NanoChat server...')
        server.terminate()
        server.wait(timeout=10)
        print('Stopped existing server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing server cleanly:', exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print('Killed existing server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web',
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--model-tag', SERVE_MODEL_TAG,
    '--num-gpus', '1',
    '--port', str(SERVER_PORT),
]

print('Starting NanoChat server:')
print(' '.join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Server process started with PID {server.pid}.')

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f'NanoChat server exited early with code {server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat server is ready: {BASE_URL}')
else:
    print(f'NanoChat server is still loading. Wait a bit, then use: {BASE_URL}')


In [ ]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')
